# Phase 1 — 파서 & Pydantic 스키마

**목표:** Claude.ai 내보내기 JSON을 내부 `session.json`으로 변환하는 파서와 Pydantic 모델을 직접 구현한다.

**학습 흐름 (각 실습마다 동일):**
```
이론 설명 → 워밍업(완성 예제 직접 실행) → 미니 실습 → 채점 → 본 실습 → 채점
```

**완성 후 연결:**
- `src/models.py` — 블록/Turn/Session 모델 붙여넣기
- `src/parser.py` — 파서 함수 붙여넣기

**AI 취업 포인트:**
> Pydantic은 FastAPI, LangChain, OpenAI Python SDK, Instructor 등
> 거의 모든 주요 AI 프레임워크의 핵심 의존성이다.
> 타입 안전한 데이터 파이프라인 설계 능력은 MLOps/AI 엔지니어 포지션에서 필수로 검토되는 역량이다.


---
## 섹션 1 — Pydantic이란? (이론)

### 왜 Pydantic인가?

AI 시스템에서 데이터는 항상 외부에서 들어온다 — JSON API, LLM 응답, 사용자 입력.  
타입 오류 하나가 잘못된 임베딩, 잘못된 RAG 결과로 이어진다.

```python
# 일반 딕셔너리: 런타임에 폭발
block = {"type": "thinking", "thinking": "...", "extra": None}
block["text"]  # KeyError — 어디서 터질지 모름

# Pydantic: 생성 시점에 검증
class ThinkingBlock(BaseModel):
    type: Literal["thinking"]
    text: str

ThinkingBlock(type="thinking", text="...")  # 성공
ThinkingBlock(type="text",     text="...")  # ValidationError — 즉시 감지
```

### 핵심 개념 3가지

| 개념 | 설명 |
|------|------|
| `BaseModel` | 모든 모델의 부모. `__init__` 자동 생성, 검증 내장 |
| `Literal["text"]` | 필드값을 특정 문자열로 고정 (타입 내로잉) |
| `model_dump()` | Pydantic 모델 → 딕셔너리. `model_dump_json()`은 JSON 문자열 |

### 실제 사용처

```
FastAPI      — 요청/응답 스키마, 자동 OpenAPI 문서 생성
LangChain    — Tool, Message, Runnable 등 모든 내부 타입
OpenAI SDK   — ChatCompletionMessage, ToolCall 등
Instructor   — LLM 응답을 Pydantic 모델로 파싱 (structured output)
이 프로젝트  — Block, Turn, Session 모델 → parser.py, server.py
```


---
## 섹션 2 — 데이터 구조 이해 (이론)

### conversations.json 최상위 구조

```json
[
  {
    "uuid": "d460d603-...",          // → session_id
    "name": "소프트웨어 아키텍처",    // → title
    "created_at": "2025-12-01T...",
    "updated_at": "2025-12-03T...",
    "chat_messages": [ ... ]         // → turns
  }
]
```

### chat_message 구조

```json
{
  "sender": "human",        // "human" | "assistant"  → "user" | "assistant"
  "content": [ ... ]        // 블록 목록
}
```

### 5가지 블록 타입

| type | 핵심 필드 | 주의사항 |
|------|----------|--------|
| `text` | `text` | 그대로 |
| `thinking` | `thinking` | 내부에서는 `text`로 정규화 |
| `tool_use` | `name`, `input` | input은 임의 딕셔너리 |
| `tool_result` | `name`, `content[]`, `is_error` | content 항목도 별도 모델 |
| `token_budget` | `remaining` | RAG 컨텍스트에서 제외 |


In [ ]:
# 데이터 탐색 — 실행해서 구조 확인
import json
from pathlib import Path

# 노트북 위치(notebooks/)에서 상위 폴더의 data/ 참조
DATA_PATH = Path("../data/conversations.json")

with open(DATA_PATH, encoding="utf-8") as f:
    conversations = json.load(f)

print(f"전체 대화 수: {len(conversations)}")
conv = conversations[0]
print(f"첫 번째 대화 키: {list(conv.keys())}")

msg = conv["chat_messages"][0]
print(f"\n메시지 sender: {msg['sender']}")
print(f"content 블록 수: {len(msg['content'])}")
print(f"첫 번째 블록 type: {msg['content'][0]['type']}")


In [ ]:
# 블록 타입 분포
from collections import Counter

block_counter = Counter(
    block["type"]
    for conv in conversations
    for msg in conv["chat_messages"]
    for block in msg["content"]
)
print("블록 타입 분포:")
for btype, count in block_counter.most_common():
    print(f"  {btype:15s}: {count:5d}개")


---
## 실습 1 — TextBlock, ThinkingBlock

**목표:** `Literal` 타입으로 필드값을 고정한 Pydantic 모델을 만든다.

**핵심 패턴:**
```python
class 모델명(BaseModel):
    type: Literal["고정값"]   # 이 필드는 항상 "고정값"만 허용
    내용: str
```


### 워밍업 — 완성 예제 실행

아래 셀을 **그대로 실행**해서 Pydantic 모델이 어떻게 동작하는지 확인한다.


In [ ]:
# 워밍업: 서버 로그 모델 (완성 코드 — 수정 말고 실행만)
from pydantic import BaseModel, ValidationError
from typing import Literal

class InfoLog(BaseModel):
    level: Literal["info"]
    message: str

class ErrorLog(BaseModel):
    level: Literal["error"]
    message: str
    code: int

# 정상 생성
log = InfoLog(level="info", message="서버 시작")
print("생성:", log)
print("level:", log.level)
print("dict:", log.model_dump())

# Literal 위반 → ValidationError
try:
    InfoLog(level="warning", message="이건 안 됨")
except ValidationError as e:
    print("\nValidationError 발생!")
    print(e.errors()[0]["msg"])


### 미니 실습 — 직접 작성

위 패턴을 참고해서 날씨 모델을 만들어보세요.

```
SunnyWeather: condition = "sunny" (고정), temperature: float
RainyWeather:  condition = "rainy" (고정), temperature: float, rainfall_mm: float
```


In [ ]:
# 미니 실습: 날씨 모델
# 힌트: InfoLog, ErrorLog 구조와 동일한 패턴

class SunnyWeather(BaseModel):
    condition: Literal['sunny']
    temperature: float


class RainyWeather(BaseModel):
    condition: Literal['rainy']
    temperature: float
    rainfall_mm: float


In [ ]:
# 미니 채점
try:
    w1 = SunnyWeather(condition="sunny", temperature=28.5)
    assert w1.condition == "sunny"
    assert w1.temperature == 28.5
    print("✓ SunnyWeather 통과")
except Exception as e:
    print(f"✗ SunnyWeather 실패: {e}")

try:
    w2 = RainyWeather(condition="rainy", temperature=18.0, rainfall_mm=12.3)
    assert w2.condition == "rainy"
    assert w2.rainfall_mm == 12.3
    # Literal 검증
    try:
        RainyWeather(condition="cloudy", temperature=20.0, rainfall_mm=0.0)
        print("✗ Literal 검증 실패")
    except ValidationError:
        print("✓ RainyWeather 통과 (Literal 검증 포함)")
except Exception as e:
    print(f"✗ RainyWeather 실패: {e}")


### 본 실습

이제 실제 블록 모델을 만든다.

```
TextBlock:     type = "text" (고정),     text: str
ThinkingBlock: type = "thinking" (고정), text: str
```

> **주의:** `ThinkingBlock`의 필드 이름은 `text`다.  
> 원본 JSON에서는 `thinking` 키지만, 파서에서 `text`로 정규화한다.


In [ ]:
# TODO: TextBlock
# 필드: type (Literal["text"]), text (str)
class TextBlock(BaseModel):
    type: Literal['text']
    text: str


# TODO: ThinkingBlock
# 필드: type (Literal["thinking"]), text (str)  ← "thinking" 아님!
class ThinkingBlock(BaseModel):
    type: Literal['thinking']
    text: str


In [ ]:
# 채점
try:
    b = TextBlock(type="text", text="안녕하세요")
    assert b.type == "text"
    assert b.text == "안녕하세요"
    assert b.model_dump() == {"type": "text", "text": "안녕하세요"}
    print("✓ TextBlock 통과")
except Exception as e:
    print(f"✗ TextBlock 실패: {e}")

try:
    b = ThinkingBlock(type="thinking", text="내 생각은...")
    assert b.type == "thinking"
    assert b.text == "내 생각은..."
    try:
        TextBlock(type="wrong", text="hi")
        print("✗ Literal 검증 실패")
    except ValidationError:
        pass
    print("✓ ThinkingBlock 통과")
except Exception as e:
    print(f"✗ ThinkingBlock 실패: {e}")


---
## 실습 2 — 중첩 모델 & 기본값

**목표:** 다른 모델을 필드로 가지는 중첩 모델과 기본값 있는 필드를 만든다.

**핵심 패턴:**
```python
from typing import Any

class Inner(BaseModel):
    value: str

class Outer(BaseModel):
    items: list[Inner]        # 중첩 모델 리스트
    data: dict[str, Any]      # 임의 딕셔너리
    flag: bool = False        # 기본값
    count: int | None = None  # Optional
```


### 워밍업 — 완성 예제 실행


In [ ]:
# 워밍업: 주문 모델 (완성 코드 — 실행만)
from typing import Any

class OrderItem(BaseModel):
    name: str
    price: float

class Order(BaseModel):
    order_id: str
    items: list[OrderItem]              # 중첩 모델 리스트
    metadata: dict[str, Any]            # 임의 딕셔너리
    is_paid: bool = False               # 기본값
    discount: float | None = None       # Optional

order = Order(
    order_id="ORD-001",
    items=[
        OrderItem(name="커피", price=4500),
        OrderItem(name="케이크", price=6000),
    ],
    metadata={"source": "mobile", "version": 2},
)
print("주문:", order.model_dump())
print("기본값 확인 — is_paid:", order.is_paid)   # False
print("기본값 확인 — discount:", order.discount)   # None
print("중첩 모델 접근:", order.items[0].name)
print("딕셔너리 접근:", order.metadata["source"])


### 미니 실습 — 직접 작성

위 패턴을 참고해서 블로그 모델을 만들어보세요.

```
Tag:     name (str)
Article: title (str), tags (list[Tag]), extra (dict[str, Any]),
         published (bool, 기본값 False), view_count (int | None, 기본값 None)
```


In [ ]:
# 미니 실습: 블로그 모델

class Tag(BaseModel):
    name: str


class Article(BaseModel):
    title: str
    tags: list[Tag]
    extra: dict[str, Any]
    published: bool = False
    view_count: int | None = None


In [ ]:
# 미니 채점
try:
    article = Article(
        title="Pydantic 입문",
        tags=[Tag(name="python"), Tag(name="pydantic")],
        extra={"author": "홍길동", "reading_time": 5},
    )
    assert article.title == "Pydantic 입문"
    assert len(article.tags) == 2
    assert article.tags[0].name == "python"
    assert article.published == False
    assert article.view_count is None
    print("✓ Article 통과")
except Exception as e:
    print(f"✗ Article 실패: {e}")


### 본 실습

**실제 데이터 예시:**
```json
{"type": "tool_use", "name": "artifacts", "input": {"id": "abc", "content": "..."}}
{"type": "tool_result", "name": "artifacts", "content": [{"type": "text", "text": "OK"}], "is_error": false}
{"type": "token_budget", "remaining": null}
{"type": "??future??", "foo": "bar"}   // → FallbackBlock
```

> FallbackBlock의 `type`은 `Literal`이 아닌 `str` — 어떤 값이든 받아야 한다.


In [ ]:
# TODO: ToolUseBlock
# 필드: type (Literal["tool_use"]), name (str), input (dict[str, Any])
class ToolUseBlock(BaseModel):
    type: Literal["tool_use"]
    name: str
    input: dict[str, Any]


# TODO: ToolResultContentItem  (tool_result content 배열의 항목)
# 필드: type (str), text (str, 기본값 "")
class ToolResultContentItem(BaseModel):
    type: str
    text: str = ""


# TODO: ToolResultBlock
# 필드: type (Literal["tool_result"]), name (str),
#        content (list[ToolResultContentItem]), is_error (bool, 기본값 False)
class ToolResultBlock(BaseModel):
    type: Literal['tool_result']
    name: str
    content: list[ToolResultContentItem]
    is_error: bool = False


# TODO: TokenBudgetBlock
# 필드: type (Literal["token_budget"]), remaining (int | None, 기본값 None)
class TokenBudgetBlock(BaseModel):
    type: Literal['token_budget']
    remaining: int | None = None


# TODO: FallbackBlock  (미래 블록 타입 보존용)
# 필드: type (str — Literal 아님), raw (dict[str, Any])
class FallbackBlock(BaseModel):
    type: str
    raw: dict[str, Any]


In [ ]:
# 채점
try:
    b = ToolUseBlock(type="tool_use", name="artifacts", input={"id": "abc"})
    assert b.name == "artifacts" and b.input["id"] == "abc"
    print("✓ ToolUseBlock 통과")
except Exception as e:
    print(f"✗ ToolUseBlock 실패: {e}")

try:
    item = ToolResultContentItem(type="text", text="OK")
    b = ToolResultBlock(type="tool_result", name="artifacts", content=[item])
    assert b.content[0].text == "OK"
    assert b.is_error == False
    b2 = ToolResultBlock(type="tool_result", name="artifacts", content=[])
    assert b2.is_error == False, "is_error 기본값 오류"
    print("✓ ToolResultBlock 통과")
except Exception as e:
    print(f"✗ ToolResultBlock 실패: {e}")

try:
    b1 = TokenBudgetBlock(type="token_budget")
    assert b1.remaining is None
    b2 = TokenBudgetBlock(type="token_budget", remaining=5000)
    assert b2.remaining == 5000
    print("✓ TokenBudgetBlock 통과")
except Exception as e:
    print(f"✗ TokenBudgetBlock 실패: {e}")

try:
    b = FallbackBlock(type="future_type", raw={"foo": "bar"})
    assert b.type == "future_type"
    assert b.raw["foo"] == "bar"
    print("✓ FallbackBlock 통과")
except Exception as e:
    print(f"✗ FallbackBlock 실패: {e}")


### 질문

1. discount: float | None = None       # Optional 의 의미

### 정리
1. 일반적인 json 파일로 받는 경우 에러가 나는 경우 잡아내기 쉽지 않음. pydentic으로 생성하는 즉시 바로 type 오류를 잡아냄. 
2. Literal 옵션으로 고정할 수 있음. 
3. 다른 모델 필드를 가지는 중첩 모델이 존재한다.
4. 기본값을 지정해줄 수 있다.
5. Any로 어떤 것이든 받을 수 있도록 설정할 수 있다.


### 실습 2 질문 — 답변

**`discount: float | None = None` 의 의미**

| 부분 | 의미 |
|------|------|
| `float` | float 값을 받을 수 있다 |
| `| None` | None(값 없음)도 허용한다 |
| `= None` | 기본값이 None — 전달 안 하면 자동으로 None |

```python
order1 = Order(order_id='ORD-001', items=[], metadata={})
# discount 미전달 → discount = None

order2 = Order(order_id='ORD-002', items=[], metadata={}, discount=9.9)
# discount = 9.9
```

Python 3.10 이전 방식: `Optional[float] = None` (from typing import Optional)  
Python 3.10+에서는 `|`로 더 간결하게 표현 가능.

AI 실무 패턴:
```python
class LLMResponse(BaseModel):
    content: str | None = None   # 응답이 없을 수 있음
    error: str | None = None     # 에러가 없을 수 있음
    usage: dict | None = None    # 토큰 사용량 (선택)
```


---
## 섹션 3 — 판별 유니온 (이론)

블록이 여러 타입이면, 어떻게 하나의 리스트에 담을 수 있을까?

```python
from typing import Union
Block = Union[TextBlock, ThinkingBlock, ToolUseBlock, ToolResultBlock, TokenBudgetBlock, FallbackBlock]

# Block 타입 변수는 6가지 중 어느 것이든 담을 수 있다
blocks: list[Block] = [
    TextBlock(type="text", text="hi"),
    ThinkingBlock(type="thinking", text="..."),
]
```

### AI 시스템에서 이 패턴이 중요한 이유

LLM 출력은 항상 타입이 다른 여러 조각으로 구성된다.  
OpenAI SDK, Anthropic SDK, LangChain 모두 같은 패턴을 사용한다.

```python
# Anthropic Python SDK 실제 코드
ContentBlock = Union[TextBlock, ToolUseBlock]
```


### 피드백

오류 없음.

---
## 실습 3 — Turn, Session 모델

**목표:** Union 타입을 필드로 가지는 중첩 모델을 만든다.

**핵심 패턴:**
```python
ItemType = Union[TypeA, TypeB]

class Container(BaseModel):
    items: list[ItemType]   # 여러 타입을 하나의 리스트에
```


### 워밍업 — 완성 예제 실행


In [ ]:
# 워밍업: 채팅 메시지 모델 (완성 코드 — 실행만)
from typing import Union

class TextMsg(BaseModel):
    kind: Literal["text"]
    content: str

class ImageMsg(BaseModel):
    kind: Literal["image"]
    url: str
    caption: str

Msg = Union[TextMsg, ImageMsg]

class ChatRoom(BaseModel):
    room_id: str
    messages: list[Msg]

room = ChatRoom(
    room_id="room-001",
    messages=[
        TextMsg(kind="text", content="안녕!"),
        ImageMsg(kind="image", url="https://...", caption="사진"),
        TextMsg(kind="text", content="잘 받았어"),
    ]
)
print("메시지 수:", len(room.messages))
print("첫 메시지 타입:", type(room.messages[0]).__name__)
print("두 번째 메시지 url:", room.messages[1].url)
print("\nJSON 직렬화:")
print(room.model_dump_json(indent=2)[:300])


### 미니 실습 — 직접 작성

위 패턴을 참고해서 플레이리스트 모델을 만들어보세요.

```
SongItem:    kind = "song" (고정),    title (str), artist (str)
PodcastItem: kind = "podcast" (고정), title (str), duration_min (int)
PlaylistItem = Union[SongItem, PodcastItem]
Playlist:    name (str), items (list[PlaylistItem])
```


In [ ]:
# 미니 실습: 플레이리스트 모델

class SongItem(BaseModel):
    kind: Literal['song']
    title: str
    artist: str

class PodcastItem(BaseModel):
    kind: Literal['podcast']
    title: str
    duration_min: int


PlaylistItem = Union[SongItem, PodcastItem]

class Playlist(BaseModel):
    name: str
    items: list[PlaylistItem]


In [ ]:
# 미니 채점
try:
    pl = Playlist(
        name="아침 루틴",
        items=[
            SongItem(kind="song", title="Morning", artist="아티스트A"),
            PodcastItem(kind="podcast", title="뉴스", duration_min=15),
        ]
    )
    assert pl.name == "아침 루틴"
    assert len(pl.items) == 2
    assert isinstance(pl.items[0], SongItem)
    assert pl.items[0].artist == "아티스트A"
    assert pl.items[1].duration_min == 15
    print("✓ Playlist 통과")
except Exception as e:
    print(f"✗ Playlist 실패: {e}")


### 본 실습

```
Block = Union[TextBlock, ThinkingBlock, ToolUseBlock,
              ToolResultBlock, TokenBudgetBlock, FallbackBlock]

Turn:    role (Literal["user", "assistant"]), blocks (list[Block])
Session: session_id, title, created_at, updated_at (모두 str), turns (list[Turn])
```


In [ ]:
Block = Union[
    TextBlock,
    ThinkingBlock,
    ToolUseBlock,
    ToolResultBlock,
    TokenBudgetBlock,
    FallbackBlock,
]


# TODO: Turn
# 필드: role (Literal["user", "assistant"]), blocks (list[Block])
class Turn(BaseModel):
    role: Literal['user', 'assistant']
    blocks: list[Block]


# TODO: Session
# 필드: session_id (str), title (str),
#        created_at (str), updated_at (str), turns (list[Turn])
class Session(BaseModel):
    session_id: str
    title: str
    created_at: str
    updated_at: str
    turns: list[Turn]


In [ ]:
# 채점
try:
    t = Turn(
        role="user",
        blocks=[TextBlock(type="text", text="질문")]
    )
    assert t.role == "user"
    assert isinstance(t.blocks[0], TextBlock)
    try:
        Turn(role="bot", blocks=[])
        print("✗ role 검증 실패")
    except ValidationError:
        pass
    print("✓ Turn 통과")
except Exception as e:
    print(f"✗ Turn 실패: {e}")

try:
    s = Session(
        session_id="abc-123",
        title="테스트",
        created_at="2025-12-01T00:00:00Z",
        updated_at="2025-12-01T01:00:00Z",
        turns=[
            Turn(role="user",      blocks=[TextBlock(type="text", text="질문")]),
            Turn(role="assistant", blocks=[TextBlock(type="text", text="답변")]),
        ]
    )
    assert s.session_id == "abc-123"
    assert len(s.turns) == 2
    j = json.loads(s.model_dump_json())
    assert j["session_id"] == "abc-123"
    print("✓ Session 통과")
    print("\nJSON 미리보기:")
    print(s.model_dump_json(indent=2)[:200] + "...")
except Exception as e:
    print(f"✗ Session 실패: {e}")


### 피드백

오류 없음.

---
## 섹션 4 — 파서 설계 (이론)

### 변환 흐름

```
conversations.json
  └── conversation[]
        ├── uuid       → session_id
        ├── name       → title
        └── chat_messages[]
              ├── sender  (human → user)
              └── content[] → parse_block() 호출
```

### 왜 if/elif 로 수동 분기하는가?

`thinking` 블록은 원본 필드명이 `thinking`이지만 내부 모델은 `text`를 사용한다.  
이 변환을 파서 계층에서만 처리하면, 나머지 코드는 항상 정규화된 모델만 다룬다.

```python
def parse_block(raw):
    t = raw.get("type")
    if t == "text":
        return TextBlock(type="text", text=raw["text"])
    elif t == "thinking":
        return ThinkingBlock(type="thinking", text=raw["thinking"])  # ← 변환!
    ...
```


---
## 실습 4 — parse_block() 함수

**목표:** `type` 값으로 분기해서 알맞은 모델을 반환하는 함수를 만든다.

**핵심 패턴:**
```python
def parse_X(raw: dict):
    t = raw.get("type")
    if t == "A":
        return ModelA(type=t, field=raw["field"])
    elif t == "B":
        return ModelB(...)
    else:
        return FallbackModel(type=t, raw=raw)
```


### 워밍업 — 완성 예제 실행


In [ ]:
# 워밍업: 동물 파서 (완성 코드 — 실행만)

class Dog(BaseModel):
    type: Literal["dog"]
    breed: str

class Cat(BaseModel):
    type: Literal["cat"]
    indoor: bool

class UnknownAnimal(BaseModel):
    type: str
    raw: dict[str, Any]

Animal = Union[Dog, Cat, UnknownAnimal]

def parse_animal(raw: dict) -> Animal:
    t = raw.get("type")
    if t == "dog":
        return Dog(type="dog", breed=raw["breed"])
    elif t == "cat":
        return Cat(type="cat", indoor=raw["indoor"])
    else:
        return UnknownAnimal(type=t, raw=raw)

# 테스트
a1 = parse_animal({"type": "dog", "breed": "Husky", "color": "white"})
a2 = parse_animal({"type": "cat", "indoor": True, "name": "나비"})
a3 = parse_animal({"type": "fish", "tank_size": 50})

print("Dog:", type(a1).__name__, a1.breed)
print("Cat:", type(a2).__name__, a2.indoor)
print("Unknown:", type(a3).__name__, a3.type, a3.raw)


### 미니 실습 — 직접 작성

위 패턴을 참고해서 도형 파서를 만들어보세요.

```
Circle:    type = "circle",    radius: float
Rectangle: type = "rectangle", width: float, height: float
OtherShape: type (str), raw (dict)
```

parse_shape(raw):
  - type이 circle → Circle 반환
  - type이 rectangle → Rectangle 반환
  - 그 외 → OtherShape 반환


In [ ]:
# 미니 실습: 도형 파서

class Circle(BaseModel):
    type: Literal['circle']
    radius: float


class Rectangle(BaseModel):
    type: Literal['rectangle']
    width: float
    height: float


class OtherShape(BaseModel):
    type: str
    raw: dict[str, Any]


Shape = Union[Circle, Rectangle, OtherShape]


def parse_shape(raw: dict) -> Shape:
    t = raw.get('type')
    if t == 'circle':
        return Circle(type='circle', radius=raw['radius'])
    elif t == 'rectangle':
        return Rectangle(type='rectangle', width=raw['width'], height=raw['height'])
    else:
        return OtherShape(type=raw['type'], raw=raw)


In [ ]:
# 미니 채점
try:
    c = parse_shape({"type": "circle", "radius": 5.0})
    assert isinstance(c, Circle)
    assert c.radius == 5.0
    print("✓ Circle 파싱 통과")
except Exception as e:
    print(f"✗ Circle 실패: {e}")

try:
    r = parse_shape({"type": "rectangle", "width": 3.0, "height": 4.0})
    assert isinstance(r, Rectangle)
    assert r.width == 3.0
    print("✓ Rectangle 파싱 통과")
except Exception as e:
    print(f"✗ Rectangle 실패: {e}")

try:
    o = parse_shape({"type": "triangle", "sides": [3, 4, 5]})
    assert isinstance(o, OtherShape)
    assert o.type == "triangle"
    print("✓ OtherShape 파싱 통과")
except Exception as e:
    print(f"✗ OtherShape 실패: {e}")


### 본 실습

> 핵심: `thinking` 블록은 `raw["thinking"]` → `ThinkingBlock(text=...)` 로 변환한다.


In [ ]:
def parse_block(raw: dict) -> Block:
    block_type = raw.get("type")

    if block_type == "text":
        return TextBlock(type='text', text=raw['text'])

    elif block_type == "thinking":
        # 힌트: raw["thinking"] → text 필드로
        return ThinkingBlock(type='thinking', text=raw['thinking'])

    elif block_type == "tool_use":
        return ToolUseBlock(type='tool_use', name=raw['name'], input=raw['input'])

    elif block_type == "tool_result":
        # 힌트: raw["content"]의 각 항목을 ToolResultContentItem으로 변환
        return ToolResultBlock(type='tool_result', name=raw['name'], content=raw['content'], is_error=raw['is_error'])

    elif block_type == "token_budget":
        return TokenBudgetBlock(type='token_budget', remaining=raw['remaining'])
    else:
        return FallbackBlock(type=raw['type'], raw=raw)


In [ ]:
# 채점
tests = [
    ({"type": "text", "text": "안녕", "citations": []},        TextBlock,        "text"),
    ({"type": "thinking", "thinking": "생각중", "summaries": []}, ThinkingBlock, "생각중"),
    ({"type": "tool_use", "name": "x", "input": {"k": 1}},    ToolUseBlock,     None),
    ({"type": "token_budget", "remaining": None},              TokenBudgetBlock, None),
    ({"type": "??future??", "foo": "bar"},                     FallbackBlock,    None),
]
all_pass = True
for raw, expected_cls, expected_text in tests:
    try:
        b = parse_block(raw)
        btype = raw['type']
        assert isinstance(b, expected_cls), f"{btype}: {type(b).__name__} != {expected_cls.__name__}"
        if expected_text:
            assert b.text == expected_text, f"text 불일치: {b.text!r}"
        print(f"✓ {btype:15s} → {type(b).__name__}")
    except Exception as e:
        btype = raw.get('type', '?')
        print(f"✗ {btype:15s} 실패: {e}")
        all_pass = False

# tool_result 별도 (content 변환 확인)
try:
    raw_result = {
        "type": "tool_result", "name": "artifacts",
        "content": [{"type": "text", "text": "OK"}],
        "is_error": False,
    }
    b = parse_block(raw_result)
    assert isinstance(b, ToolResultBlock)
    assert b.content[0].text == "OK"
    print(f"✓ tool_result     → {type(b).__name__} (content 변환 확인)")
except Exception as e:
    print(f"✗ tool_result 실패: {e}")
    all_pass = False

if all_pass:
    print("\n✓✓✓ parse_block() 전체 통과!")


### 질문

1. 왜 other의 경우에는 raw에서 raw의 value를 뽑지 않고 그대로 raw를 출력하는지
2. 본 실습에서 다음이 무슨 말인지 잘 이해가 안감
    - 힌트: raw["thinking"] → text 필드로
    - 힌트: raw["content"]의 각 항목을 ToolResultContentItem으로 변환
3. 본 실습 결과 왜 실패인지 모르겠음.
    - TecxtBlock이 아래와 같이 생겼으므로 `return TextBlock(type='text', text=raw['text'])`로 짜야 할 것 같은데, 실패로 뜸.

    - ```python
        # TODO: TextBlock
        # 필드: type (Literal["text"]), text (str)
        class TextBlock(BaseModel):
            type: Literal['text']
            text: str
        ```

In [ ]:
# 정답: parse_block()
# 개선 포인트:
#   1. text 테스트 expected_text 버그 수정 ("text" → "안녕")
#   2. tool_result content 명시적 변환
#   3. is_error / remaining 은 .get()으로 안전하게

def parse_block_solution(raw: dict) -> Block:
    block_type = raw.get('type')

    if block_type == 'text':
        return TextBlock(type='text', text=raw['text'])

    elif block_type == 'thinking':
        return ThinkingBlock(type='thinking', text=raw['thinking'])

    elif block_type == 'tool_use':
        return ToolUseBlock(type='tool_use', name=raw['name'], input=raw['input'])

    elif block_type == 'tool_result':
        content = [ToolResultContentItem(**item) for item in raw.get('content', [])]
        return ToolResultBlock(
            type='tool_result',
            name=raw['name'],
            content=content,
            is_error=raw.get('is_error', False),
        )

    elif block_type == 'token_budget':
        return TokenBudgetBlock(type='token_budget', remaining=raw.get('remaining'))

    else:
        return FallbackBlock(type=block_type, raw=raw)


# 버그 수정된 테스트
tests_fixed = [
    ({"type": "text",         "text": "안녕",    "citations": []},  TextBlock,        "안녕"),
    ({"type": "thinking",     "thinking": "생각중", "summaries": []}, ThinkingBlock,    "생각중"),
    ({"type": "tool_use",     "name": "x",       "input": {"k": 1}}, ToolUseBlock,    None),
    ({"type": "token_budget", "remaining": None},                     TokenBudgetBlock, None),
    ({"type": "??future??",   "foo": "bar"},                          FallbackBlock,    None),
]
for raw, expected_cls, expected_text in tests_fixed:
    b = parse_block_solution(raw)
    btype = raw['type']
    assert isinstance(b, expected_cls), f"{btype} 타입 불일치"
    if expected_text:
        assert b.text == expected_text, f"text 불일치: {b.text!r}"
    print(f"✓ {btype:15s} → {type(b).__name__}")

raw_result = {
    "type": "tool_result", "name": "artifacts",
    "content": [{"type": "text", "text": "OK"}],
    "is_error": False,
}
b = parse_block_solution(raw_result)
assert isinstance(b, ToolResultBlock) and b.content[0].text == "OK"
print(f"✓ tool_result     → {type(b).__name__}")
print("\n✓✓✓ parse_block_solution 전체 통과!")


### 실습 4 질문 — 답변

**Q1. FallbackBlock에서 왜 raw 전체를 저장하는지**

미래 블록 타입이 어떤 필드를 가질지 모른다.  
특정 value만 뽑으면 나머지 데이터가 영구 손실된다.

```python
# 나쁜 예 — 나머지 필드 손실
FallbackBlock(type=t, value=raw.get('value'))

# 좋은 예 — 전체 보존, 나중에 어떤 필드든 꺼낼 수 있음
FallbackBlock(type=t, raw=raw)
```

---

**Q2. 힌트 설명**

`raw["thinking"] → text 필드로`

```python
# 원본 JSON 키:  thinking
# 내부 모델 필드: text
# → 이름이 다르므로 명시적으로 변환 필요

raw = {"type": "thinking", "thinking": "내 생각"}
ThinkingBlock(type='thinking', text=raw['thinking'])  # raw['thinking'] → text
```

`raw["content"]의 각 항목을 ToolResultContentItem으로 변환`

```python
# raw['content'] = [{'type': 'text', 'text': 'OK'}, ...]  (딕셔너리 리스트)
# 각 딕셔너리를 Pydantic 모델로 변환:
content = [ToolResultContentItem(**item) for item in raw['content']]
```

네 코드에서 `content=raw['content']`로 딕셔너리 리스트를 그대로 넣었는데 통과된 건  
Pydantic이 자동 변환해줬기 때문. 실무에서는 명시적 변환이 더 안전하다.

---

**Q3. text 블록 실패 이유**

**네 코드가 맞다. 내가 만든 테스트 케이스에 버그가 있었다.**

```python
# 버그 있는 테스트
({"type": "text", "text": "안녕", "citations": []}, TextBlock, "text"),
#                                                                    ↑ 이게 잘못됨
# expected_text = "text" 로 설정해서 b.text == "text" 를 체크함
# 그런데 b.text 는 실제로 "안녕" 이라서 불일치 → 실패

# 올바른 테스트
({"type": "text", "text": "안녕", "citations": []}, TextBlock, "안녕"),
#                                                                    ↑ 이렇게 해야 함
```

아래 정답 셀에서 테스트를 수정했다.


---
## 실습 5 — parse_conversation() 함수

**목표:** 딕셔너리를 읽어 Pydantic 모델로 변환하는 함수를 만든다.

**핵심 패턴:**
```python
def parse_X(raw: dict) -> ModelX:
    return ModelX(
        field_a = raw["key_a"],          # 키 이름이 같을 때
        field_b = raw["different_key"],  # 키 이름이 다를 때
        items   = [parse_item(i) for i in raw["items"]],  # 중첩 리스트
    )
```


### 워밍업 — 완성 예제 실행


In [ ]:
# 워밍업: 이벤트 로그 파서 (완성 코드 — 실행만)

class LogEntry(BaseModel):
    level: Literal["info", "error"]
    message: str

class EventLog(BaseModel):
    log_id: str
    service: str
    entries: list[LogEntry]

def parse_entry(raw: dict) -> LogEntry:
    return LogEntry(
        level=raw["severity"],   # severity → level 로 키 이름 변환
        message=raw["msg"],
    )

def parse_event_log(raw: dict) -> EventLog:
    return EventLog(
        log_id=raw["id"],
        service=raw["svc_name"],
        entries=[parse_entry(e) for e in raw["records"]],
    )

raw_log = {
    "id": "log-001",
    "svc_name": "auth-service",
    "records": [
        {"severity": "info",  "msg": "로그인 성공"},
        {"severity": "error", "msg": "토큰 만료"},
    ]
}

log = parse_event_log(raw_log)
print("log_id:", log.log_id)
print("service:", log.service)
print("entries:", [(e.level, e.message) for e in log.entries])


### 미니 실습 — 직접 작성

위 패턴을 참고해서 상품 파서를 만들어보세요.

```
Category:  name (str)
Product:   product_id (str), product_name (str), categories (list[Category])

raw 딕셔너리 구조:
  raw["pid"]        → product_id
  raw["label"]      → product_name
  raw["tags"]       → categories (각 항목: {"tag": "..."})
```


In [ ]:
# 미니 실습: 상품 파서

class Category(BaseModel):
    name: str


class Product(BaseModel):
    product_id: str
    product_name: str
    categories: list[Category]


def parse_category(raw: dict) -> Category:
    return Category(name=raw['tag'])


def parse_product(raw: dict) -> Product:
    return Product(product_id=raw['pid'], product_name=raw-'label', categories=[parse_category(e) for e in raw['tags']])


In [ ]:
# 미니 채점
try:
    raw_product = {
        "pid": "P-001",
        "label": "무선 키보드",
        "tags": [{"tag": "전자기기"}, {"tag": "주변기기"}]
    }
    p = parse_product(raw_product)
    assert p.product_id == "P-001"
    assert p.product_name == "무선 키보드"
    assert len(p.categories) == 2
    assert p.categories[0].name == "전자기기"
    print("✓ Product 파서 통과")
except Exception as e:
    print(f"✗ Product 파서 실패: {e}")


### 질문

1. 실패 이유를 모르겠음.

In [ ]:
# 정답: parse_product()

def parse_category_solution(raw: dict) -> Category:
    return Category(name=raw['tag'])


def parse_product_solution(raw: dict) -> Product:
    return Product(
        product_id=raw['pid'],
        product_name=raw['label'],     # raw-'label' → raw['label']
        categories=[parse_category_solution(e) for e in raw['tags']],
    )


# 검증
raw_product = {
    'pid': 'P-001',
    'label': '무선 키보드',
    'tags': [{'tag': '전자기기'}, {'tag': '주변기기'}]
}
p = parse_product_solution(raw_product)
assert p.product_id == 'P-001'
assert p.product_name == '무선 키보드'
assert p.categories[0].name == '전자기기'
print('✓ parse_product_solution 통과')


### 실습 5 미니 질문 — 답변

**실패 이유: 오타**

```python
# 틀린 코드
product_name=raw-'label'   # dict - str 은 Python에 없는 연산

# 올바른 코드
product_name=raw['label']  # 대괄호로 키 접근
```

딕셔너리 값을 꺼낼 때는 항상 `raw['키']` 또는 `raw.get('키')`를 사용한다.


### 본 실습

**변환 규칙:**
```
conv["uuid"]          → session_id
conv["name"]          → title
conv["chat_messages"] → turns

msg["sender"]: "human" → "user",  "assistant" → "assistant"
msg["content"]        → parse_block() 목록
```


In [ ]:
SENDER_TO_ROLE = {"human": "user", "assistant": "assistant"}


def parse_message(msg: dict) -> Turn:
    # TODO: sender→role 변환, content 블록 목록 변환
    role = SENDER_TO_ROLE[msg['sender']]
    blocks = [parse_block(b) for b in msg['content']]
    return Turn(role=role, blocks=blocks)


def parse_conversation(conv: dict) -> Session:
    # TODO: uuid→session_id, name→title, chat_messages→turns
    session_id = conv['uuid']
    title = conv['name']
    turns = [parse_message(t) for t in conv['chat_messages']]
    return Session(session_id=session_id, title=title, turns = turns, created_at=conv['created_at'], updated_at=conv['updated_at'])


In [ ]:
# 정답: parse_message(), parse_conversation()
SENDER_TO_ROLE = {"human": "user", "assistant": "assistant"}


def parse_message_solution(msg: dict) -> Turn:
    role = SENDER_TO_ROLE[msg['sender']]                          # human → user
    blocks = [parse_block_solution(b) for b in msg['content']]
    return Turn(role=role, blocks=blocks)


def parse_conversation_solution(conv: dict) -> Session:
    return Session(
        session_id=conv['uuid'],                                   # uuid → session_id
        title=conv['name'],                                        # name → title
        created_at=conv['created_at'],
        updated_at=conv['updated_at'],
        turns=[parse_message_solution(m) for m in conv['chat_messages']],
    )


# 검증
s = parse_conversation_solution(conversations[0])
assert isinstance(s, Session)
assert s.session_id == conversations[0]['uuid']
assert s.title == conversations[0]['name']
assert len(s.turns) == len(conversations[0]['chat_messages'])
print(f'✓ session_id: {s.session_id}')
print(f'✓ title:      {s.title}')
print(f'✓ turns:      {len(s.turns)}개')
print(f'✓ 첫 turn role: {s.turns[0].role}')


In [ ]:
# 채점: 실제 데이터로 검증
session = parse_conversation(conversations[0])

assert isinstance(session, Session), f"Session이어야 함, 실제: {type(session)}"
assert session.session_id == conversations[0]["uuid"]
assert session.title == conversations[0]["name"]
assert len(session.turns) == len(conversations[0]["chat_messages"])

first_msg  = conversations[0]["chat_messages"][0]
first_turn = session.turns[0]
expected_role = SENDER_TO_ROLE[first_msg["sender"]]
assert first_turn.role == expected_role, f"role 불일치: {first_turn.role!r}"
assert len(first_turn.blocks) == len(first_msg["content"]), "블록 수 불일치"

print(f"✓ session_id: {session.session_id}")
print(f"✓ title: {session.title}")
print(f"✓ turns: {len(session.turns)}개")
print(f"✓ 첫 turn role: {session.turns[0].role}")

dist = Counter(type(b).__name__ for t in session.turns for b in t.blocks)
print("\n블록 분포:")
for btype, cnt in dist.most_common():
    print(f"  {btype}: {cnt}")


### 피드백

#### 실습 4
버그 3개:

① 함수 정의 타이포 (line 186)

`cldef parse_block(raw: dict) -> Block:  # ← cl이 앞에 붙음`

→ `def parse_block(...)` 이어야 함. 노트북에서 실행하면 SyntaxError 났을 거야.


② tool_result — `is_error` KeyError 위험 (line 201)

`is_error=raw['is_error']         # 없으면 KeyError`
`is_error=raw.get('is_error', False)  # 방어적 접근`


③ token_budget — remaining KeyError 위험 (line 204)


`remaining=raw['remaining']       # 없으면 KeyError`
`remaining=raw.get('remaining')   # 방어적 접근`
나머지 분기는 전부 정확.

#### 실습 5 
버그 1개 (미니 실습, line 230):


`product_name=raw-'label'    # ← - 연산자 (이전에도 나온 타이포)`
`product_name=raw['label']   # 딕셔너리 접근은 []`
`parse_message`, `parse_conversation` 본 실습은 정확.

---
## 최종 통합 — 전체 데이터셋 파싱

224개 대화 전체를 파싱해 오류 없이 통과하면 Phase 1 완료.


In [ ]:
errors = []
block_type_dist = Counter()

for i, conv in enumerate(conversations):
    try:
        session = parse_conversation(conv)
        for turn in session.turns:
            for block in turn.blocks:
                block_type_dist[type(block).__name__] += 1
    except Exception as e:
        errors.append((i, conv.get("uuid"), str(e)))

print(f"전체: {len(conversations)}개  |  성공: {len(conversations)-len(errors)}  |  실패: {len(errors)}")

if errors:
    print("\n실패 목록:")
    for idx, uid, err in errors[:5]:
        print(f"  [{idx}] {uid}: {err}")
else:
    print("\n✓ 모든 대화 파싱 성공!")

print("\n블록 타입 분포:")
for btype, cnt in block_type_dist.most_common():
    print(f"  {btype:30s}: {cnt:5d}개")


---
## 연결 — 다음 단계

### 구현한 것 → .py 파일로 옮기기

| 구현 | 파일 |
|------|------|
| 블록 모델 6개, Turn, Session | `src/models.py` |
| parse_block, parse_message, parse_conversation | `src/parser.py` |

### Phase 2 연결

```
python -m parser --input data/conversations.json
  → parse_export() 호출
  → conversations/{session_id}.json 저장

python -m viewer
  → GET /api/sessions → session.json 목록 반환
  → 뷰어(app.js)에서 turns[].blocks[] 렌더링
```

### AI 취업 연결

```
OpenAI 응답 파싱     → choices[].message.content[]
Anthropic 응답 파싱  → message.content[] (TextBlock | ToolUseBlock)
LangChain 문서 파싱  → Document(page_content, metadata)
```

모두 외부 JSON → Pydantic 모델 패턴이다.  
discriminated union + 변환 계층이 표준 해법이다.
